# IPA – worked example
Compute the Water Potability Index for the Barcarena dataset, inspect components and coverage, and test robustness.

In [ ]:
import pandas as pd, sys
sys.path.insert(0, '..')
import ipa
df = pd.read_csv('../data/barcarena_2018_wells.csv')
W = pd.read_csv('../data/final_weights.csv', index_col=0)
res = ipa.ipa(df, W['health'])
res['class'].value_counts().reindex(ipa.CLASSES)

In [ ]:
# components of the five worst samples
res.sort_values('IPA', ascending=False)[['IPA','IPA_PC','IPA_MB','IPA_HM','coverage','class']].head()

In [ ]:
# missing data: drop all metals - the index is still computable, coverage falls
no_metals = df.drop(columns=ipa.COMPONENTS['HM'][:-1])
r2 = ipa.ipa(no_metals, W['health'])
r2[['IPA','coverage','class','partial']].head()

In [ ]:
# your own lens: change tiers and recompute goal-programming weights
X = ipa.excursion(df)[ipa.P23]
my_lens = ipa.LENSES['health']
w_new = ipa.gp_weights(my_lens, ipa.critic(X), lam=0.5)
ipa.ipa(df, w_new)['class'].value_counts().reindex(ipa.CLASSES)

In [ ]:
# robustness: class acceptability under +/-30% weight perturbation
acc = ipa.smaa(df, W['health'])
(acc.max(axis=1) >= 0.95).sum()